# 03 - Quy trình làm việc chuẩn với Qiskit 2.x

Gom lại thành 5 bước dùng cho mọi bài toán:

1. **Xây mạch** - `QuantumCircuit`, các cổng, thanh ghi
2. **Xem mạch** - `draw()`, `depth()`, `count_ops()`
3. **Đo** - `measure()` hoặc `measure_all()`
4. **Chạy** - `Statevector` / `SamplerV2` / `EstimatorV2`
5. **Transpile** - dịch mạch về tập cổng mà phần cứng hiểu

API Qiskit thay đổi khá nhiều từ 0.x sang 2.x, nên phần cuối notebook có bảng đối chiếu
API cũ -> API hiện tại để bạn khỏi bị lạc khi đọc tài liệu cũ trên mạng.

In [ ]:
import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2, SamplerV2
from IPython.display import display

import qiskit
import qiskit_aer

print("qiskit:", qiskit.__version__, "| qiskit-aer:", qiskit_aer.__version__)

## Bước 1 - Xây mạch

Hai cách khai báo:

- `QuantumCircuit(2, 2)` - ngắn gọn; thanh ghi cổ điển tự động tên `"c"`.
- `QuantumCircuit(QuantumRegister(...), ClassicalRegister(...))` - đặt tên rõ ràng.
  **Tên thanh ghi chính là tên bạn dùng để đọc kết quả**, nên đặt tên tử tế sẽ đỡ nhầm sau này.

`barrier()` không phải cổng vật lý - nó chỉ chặn tối ưu hóa và làm bản vẽ dễ đọc.

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.barrier()
print(qc.draw())

qr = QuantumRegister(2, "dulieu")
cr = ClassicalRegister(2, "ketqua")
qc_named = QuantumCircuit(qr, cr)
qc_named.h(qr[0])
qc_named.measure(qr, cr)
print(qc_named.draw())
print("tên thanh ghi cổ điển:", qc_named.cregs[0].name)

## Bước 2 - Xem và đo kích thước mạch

- `depth()` - đường dài nhất qua mạch (số "lớp"); ảnh hưởng trực tiếp tới lỗi do decoherence.
- `size()` - tổng số cổng.
- `count_ops()` - đếm từng loại cổng. Nhìn vào số `cx` là biết mạch đắt hay rẻ.

`draw("mpl")` vẽ bằng matplotlib (cần `pylatexenc`, đã có sẵn trong venv này).

In [ ]:
print("depth    :", qc.depth())
print("size     :", qc.size())
print("count_ops:", dict(qc.count_ops()))

fig = qc.draw("mpl")
display(fig)
fig.savefig("03_mach.png", dpi=150, bbox_inches="tight")
print("đã lưu 03_mach.png")

## Bước 3 - Đo: `measure()` hay `measure_all()`

| Cách viết | Thanh ghi kết quả | Đọc bằng |
|---|---|---|
| `QuantumCircuit(2, 2)` + `qc.measure([0,1],[0,1])` | `c` (mặc định) | `res.data.c` |
| `QuantumCircuit(2)` + `qc.measure_all()` | `meas` (tự tạo) | `res.data.meas` |

`measure_all()` tiện hơn vì không cần khai báo bit cổ điển, nhưng nhớ là nó **tự thêm barrier**
và đặt tên thanh ghi là `meas`. Đọc nhầm tên thanh ghi là lỗi rất hay gặp.

In [ ]:
a = QuantumCircuit(2, 2)
a.h(0)
a.cx(0, 1)
a.measure([0, 1], [0, 1])

b = QuantumCircuit(2)
b.h(0)
b.cx(0, 1)
b.measure_all()

ra = SamplerV2().run([a], shots=512).result()[0]
rb = SamplerV2().run([b], shots=512).result()[0]

print("measure()     -> ", ra.data, "->", ra.data.c.get_counts())
print("measure_all() -> ", rb.data, "->", rb.data.meas.get_counts())

## Bước 4 - Ba cách lấy kết quả

| Cách | Cho ra gì | Mạch có được phép có `measure`? | Dùng khi nào |
|---|---|---|---|
| `Statevector(qc)` | amplitude chính xác, không nhiễu | **Không** | học / debug mạch nhỏ |
| `SamplerV2` | counts từ các lần đo | **Có, bắt buộc** | mô phỏng kết quả thật |
| `EstimatorV2` | expectation value $\langle O\rangle$ | **Không** | VQE, đo tương quan |

Nhớ: `Statevector` chỉ chạy được với mạch **không có** lệnh đo, còn `Sampler` thì **phải có**.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

# (a) chính xác
print("(a) Statevector :", np.round(Statevector(bell).data, 3))

# (b) lấy mẫu
bell_do = bell.copy()
bell_do.measure_all()
counts = SamplerV2().run([bell_do], shots=1024).result()[0].data.meas.get_counts()
print("(b) SamplerV2   :", counts)

# (c) expectation value
ev = EstimatorV2().run([(bell, SparsePauliOp("ZZ"))]).result()[0].data.evs
print("(c) EstimatorV2 : <ZZ> =", round(float(ev), 3))

## Bước 5 - Transpile: vì sao không bỏ qua được

Phần cứng thật (và cả simulator khi bị giới hạn) chỉ hiểu một tập **basis gates** nhỏ, ví dụ
`["rz", "sx", "x", "cx"]`, và chỉ cho phép `cx` giữa các cặp qubit **nối trực tiếp với nhau**
(coupling map). `transpile()` làm hai việc:

1. **Dịch** mọi cổng về tập basis gates.
2. **Routing**: chèn `swap` để đưa hai qubit cần tương tác lại gần nhau.

`optimization_level` từ 0 đến 3 (Qiskit 2.x mặc định **2**): càng cao càng tối ưu kỹ,
bù lại dịch lâu hơn.
Ở dưới là mạch có gate thừa (`x` hai lần, `h` hai lần) và một `cx(0, 2)` trong khi máy chỉ nối
0-1-2, để thấy rõ khác biệt.

In [ ]:
tho = QuantumCircuit(3)
tho.h(0)
tho.x(1)
tho.x(1)  # hai cổng X liên tiếp = không làm gì
tho.cx(0, 2)  # nhưng máy chỉ nối 0-1 và 1-2
tho.h(0)
tho.h(0)
tho.measure_all()

print("mạch gốc  : depth", tho.depth(), dict(tho.count_ops()))
print(tho.draw())

for muc in range(4):
    t = transpile(
        tho,
        basis_gates=["rz", "sx", "x", "cx"],
        coupling_map=[[0, 1], [1, 2]],
        optimization_level=muc,
        seed_transpiler=11,
    )
    print(f"optimization_level={muc}: depth {t.depth():3d}  {dict(t.count_ops())}")

In [ ]:
t = transpile(
    tho,
    basis_gates=["rz", "sx", "x", "cx"],
    coupling_map=[[0, 1], [1, 2]],
    optimization_level=0,
    seed_transpiler=11,
)
print("Mức 0 - thấy rõ các cổng đã bị dịch và swap được chèn vào:")
print(t.draw(fold=110))

### Đường chạy trực tiếp qua backend

Ngoài primitive, vẫn có thể gọi thẳng `backend.run()`. Khi đó phải `transpile` trước vì
backend không tự dịch mạch hộ bạn. `AerSimulator` hiểu sẵn rất nhiều cổng nên mạch đơn giản
vẫn chạy được nếu bạn bỏ qua bước này - nhưng với phần cứng thật thì **bắt buộc**.

In [ ]:
backend = AerSimulator()
mach_da_dich = transpile(bell_do, backend)
ket_qua = backend.run(mach_da_dich, shots=1024).result()
print("counts:", ket_qua.get_counts())
print("backend hiểu:", len(backend.target.operation_names), "loại cổng")

## Parameter - một mạch, nhiều góc

Không cần dựng lại mạch cho từng góc. Khai báo `Parameter` rồi:

- `assign_parameters({...})` để gán giá trị, hoặc
- gửi thẳng nhiều bộ giá trị cho Sampler qua "pub": `sampler.run([(mach, danh_sach_goc)])`.

Cách thứ hai nhanh hơn nhiều khi quét tham số (ví dụ khi huấn luyện variational circuit).

In [ ]:
theta = Parameter("theta")
pqc = QuantumCircuit(1)
pqc.ry(theta, 0)
pqc.measure_all()

goc = [[0.0], [np.pi / 2], [np.pi]]
res = SamplerV2().run([(pqc, goc)], shots=512).result()[0]

for i, g in enumerate(goc):
    print(f"theta = {g[0]:.4f} -> {res.data.meas.get_counts(i)}")

## Tái sử dụng mạch: `compose` và `to_gate`

- `compose(...)` - dán mạch con vào mạch lớn, các cổng nằm "phẳng" chung một mức.
- `append(mach.to_gate(), qubits)` - đóng gói thành một khối; bản vẽ gọn hơn, muốn xem
  bên trong thì gọi `decompose()`.

In [ ]:
khoi_bell = bell.to_gate()
khoi_bell.name = "bell"

big = QuantumCircuit(4)
big.compose(bell, qubits=[0, 1], inplace=True)  # dán phẳng
big.append(khoi_bell, [2, 3])  # đóng gói thành một khối
print(big.draw())

print("count_ops           :", dict(big.count_ops()))
print("sau khi decompose() :", dict(big.decompose().count_ops()))

## API cũ (Qiskit 0.x) vs API hiện tại

Rất nhiều tutorial trên mạng vẫn dùng API cũ đã bị gỡ bỏ. Bảng đối chiếu:

| Cũ | Hiện tại |
|---|---|
| `from qiskit import execute` | đã gỡ bỏ - dùng primitive hoặc `backend.run()` |
| `execute(qc, backend, shots=1024)` | `SamplerV2().run([qc], shots=1024)` |
| `Aer.get_backend("qasm_simulator")` | `AerSimulator()` từ `qiskit_aer` |
| `result.get_counts()` | `result()[0].data.<ten_thanh_ghi>.get_counts()` |
| `qiskit.providers.aer` | package riêng `qiskit-aer` |
| gửi mạch thẳng tới backend | `transpile(qc, backend)` rồi mới `backend.run()` |

### Những lỗi hay gặp

1. **Đọc nhầm thanh ghi** - `measure_all()` tạo thanh ghi `meas`, không phải `c`.
2. **Gọi `Statevector(qc)` khi mạch đã có `measure`** - phải bỏ lệnh đo đi.
3. **Quên `transpile`** trước `backend.run()`.
4. **Nhầm thứ tự bit** - qubit 0 là ký tự bên phải (`'001'` = qubit 0 đang bật).
5. **So sánh cổng bằng `==` trong khi chỉ cần `equiv`** - khác nhau ở global phase.

## Bước tiếp theo

Bạn đã có đủ nền tảng: trạng thái, cổng 1 qubit, cổng nhiều qubit, và quy trình chạy.
Hướng đi tiếp tự nhiên:

- **Thuật toán cơ bản**: Deutsch-Jozsa, Bernstein-Vazirani, Grover.
- **Mô phỏng có nhiễu**: `qiskit_aer.noise` - thêm noise model để thấy kết quả lệch đi.
- **Chạy trên máy thật**: `qiskit-ibm-runtime` (chưa cài trong venv này).